# 01 — Test Configuration & First GRPO Training

Questo notebook verifica passo-passo che tutto il setup funzioni:

1. **Imports & GPU** — verifica che torch veda la GPU
2. **Dataset** — genera (o carica) il dataset sintetico
3. **Model** — carica il modello con LoRA + 4-bit
4. **Rewards** — test rapido delle reward functions
5. **Inference test** — genera un paio di completions e valuta
6. **GRPO Training** — avvia un mini-train (pochi step)

> **Colab**: per usare su Colab, aggiungi una cella iniziale con:
> ```python
> !pip install torch transformers trl peft datasets accelerate bitsandbytes pyyaml tqdm
> ```
> e clona il repo.

In [4]:
!pip install torch transformers trl peft datasets accelerate bitsandbytes pyyaml tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00:00:0100:01


In [9]:
!git clone https://github.com/GiuseppeBellamacina/tris.git

Cloning into 'tris'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 71 (delta 14), reused 69 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 229.31 KiB | 1.76 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [16]:
%cd tris

/content/tris


In [17]:
%ls

CONTRIBUTING.md  figures/         notebooks/      src/
docs/            INSTRUCTIONS.md  pyproject.toml  tests/
experiments/     LICENSE          README.md       uv.lock


## 1. Imports & GPU Check

In [18]:
import sys
from pathlib import Path

# Assicurati che la root del progetto sia nel path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

Project root: /content/tris


In [19]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Nessuna GPU trovata! Il training sarà molto lento.")

PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB


In [20]:
import accelerate
import peft
import transformers
import trl

import datasets as hf_datasets

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"peft:         {peft.__version__}")
print(f"datasets:     {hf_datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print("\nTutti gli import OK ✓")

transformers: 5.0.0
trl:          0.29.1
peft:         0.18.1
datasets:     4.0.0
accelerate:   1.13.0

Tutti gli import OK ✓


## 2. Genera / Carica Dataset Sintetico

In [21]:
import os
from pathlib import Path

from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

os.chdir(ROOT)  # assicurati di essere nella root

DATASET_PATH = ROOT / "data" / "synthetic"

if not DATASET_PATH.exists():
    print("Dataset non trovato, lo genero...")
    ds = generate_dataset(num_samples=500, seed=42)  # 500 per test veloce
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")
else:
    print(f"Dataset trovato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))
print(f"\nSplits: {list(ds.keys())}")
for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    print(f"    columns: {split_ds.column_names}")

Dataset non trovato, lo genero...


Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset salvato in /content/tris/data/synthetic

Splits: ['train', 'test']
  train: 400 samples
    columns: ['system_prompt', 'prompt', 'task_type', 'difficulty']
  test: 100 samples
    columns: ['system_prompt', 'prompt', 'task_type', 'difficulty']


In [22]:
# Guarda qualche sample
train_ds = ds["train"]
for i in range(3):
    sample = train_ds[i]
    print(f"--- Sample {i} ---")
    print(f"  task_type:  {sample['task_type']}")
    print(f"  difficulty: {sample['difficulty']}")
    print(f"  prompt:     {sample['prompt'][:120]}...")
    print()

--- Sample 0 ---
  task_type:  python
  difficulty: hard
  prompt:     Write a Python context manager class called `ResourcePool` that wraps a database transaction with automatic commit/rollb...

--- Sample 1 ---
  task_type:  json
  difficulty: hard
  prompt:     Generate a deeply nested JSON object (at least 4 levels of nesting) representing a file system directory tree hierarchy....

--- Sample 2 ---
  task_type:  python
  difficulty: simple
  prompt:     Write a Python function `to_uppercase` that converts a temperature in Celsius to a flat list....



## 3. Carica Modello + LoRA + 4-bit

In [23]:
from src.datasets.dataloader import load_config

config = load_config(str(ROOT / "experiments" / "configs" / "grpo.yaml"))
print("Config caricata:")
for section, values in config.items():
    print(f"  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"    {k}: {v}")

Config caricata:
  [model]
    name: Qwen/Qwen2.5-Coder-1.5B-Instruct
    quantization: 4bit
    dtype: bfloat16
  [lora]
    r: 16
    lora_alpha: 32
    lora_dropout: 0.05
    target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
    task_type: CAUSAL_LM
  [dataset]
    path: data/synthetic
    split: train
    max_samples: None
  [training]
    max_steps: 1000
    per_device_train_batch_size: 1
    gradient_accumulation_steps: 8
    learning_rate: 1e-06
    lr_scheduler_type: cosine
    warmup_steps: 50
    bf16: True
    logging_steps: 10
    save_steps: 200
    save_total_limit: 3
    output_dir: experiments/checkpoints/grpo
    log_dir: experiments/logs/grpo
  [grpo]
    num_generations: 8
    max_completion_length: 512
    max_prompt_length: 256
    beta: 0.04
    temperature: 0.7
  [reward]
    type: combined
    partial_credit: False
    reasoning_bonus: 0.0
  [wandb]
    project: grpo-strict-generation
    run_name: None
    tags: ['grpo', 'Qwen2.5-Coder-1.5B']


In [24]:
from src.models.model_loader import load_model_and_tokenizer

print(f"Caricamento modello: {config['model']['name']}")
print("(questo può richiedere qualche minuto al primo download...)\n")

model, tokenizer = load_model_and_tokenizer(config)

# Info modello
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nParametri totali:    {total:,}")
print(f"Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Tokenizer vocab:     {len(tokenizer)}")
print(f"Pad token:           {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

Caricamento modello: Qwen/Qwen2.5-Coder-1.5B-Instruct
(questo può richiedere qualche minuto al primo download...)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Parametri totali:    892,974,592
Parametri trainable: 4,358,144 (0.49%)
Tokenizer vocab:     151665
Pad token:           <|endoftext|> (id=151643)


## 4. Test Reward Functions

In [25]:
from src.training.rewards import (
    combined_reward,
    json_reward,
    python_reward,
    reasoning_reward,
)

# Test JSON reward
valid_json = '```json\n{"name": "test", "age": 25, "active": true}\n```'
invalid_json = '```json\n{"name": "test", age: 25}\n```'
no_json = "Non ho generato nessun JSON."

print("=== JSON Reward ===")
print(f"  Valid JSON:   {json_reward(valid_json)}")   # atteso: 1.0
print(f"  Invalid JSON: {json_reward(invalid_json)}") # atteso: 0.0
print(f"  No JSON:      {json_reward(no_json)}")      # atteso: 0.0

# Test Python reward
valid_py = '```python\ndef add(a, b):\n    return a + b\n```'
invalid_py = '```python\ndef add(a, b)\n    return a + b\n```'  # manca :
no_py = "Ecco la mia risposta senza codice."

print("\n=== Python Reward ===")
print(f"  Valid Python:   {python_reward(valid_py)}")    # atteso: 1.0
print(f"  Invalid Python: {python_reward(invalid_py)}")  # atteso: 0.0
print(f"  No Python:      {python_reward(no_py)}")       # atteso: 0.0

# Test reasoning bonus
with_think = "<think>Devo creare un JSON con tre campi come richiesto.</think>\n" + valid_json
without_think = valid_json

print("\n=== Reasoning Reward ===")
print(f"  With <think>:    {reasoning_reward(with_think)}")    # atteso: 0.2
print(f"  Without <think>: {reasoning_reward(without_think)}") # atteso: 0.0

# Test combined
print("\n=== Combined Reward ===")
print(f"  JSON valid:           {combined_reward(valid_json, 'json')}")  # 1.0
r = combined_reward(with_think, 'json', reasoning_bonus=0.2)
print(f"  JSON valid + reason:  {r}")  # 1.2

print("\nReward functions OK ✓")

=== JSON Reward ===
  Valid JSON:   1.0
  Invalid JSON: 0.0
  No JSON:      0.0

=== Python Reward ===
  Valid Python:   1.0
  Invalid Python: 0.0
  No Python:      0.0

=== Reasoning Reward ===
  With <think>:    0.2
  Without <think>: 0.0

=== Combined Reward ===
  JSON valid:           1.0
  JSON valid + reason:  1.2

Reward functions OK ✓


## 5. Test Inference (pre-training)

In [26]:
from src.datasets.dataloader import format_prompt_for_model

# Prendi 4 sample misti
test_indices = [0, 1, 10, 20]
test_samples = [train_ds[i] for i in test_indices if i < len(train_ds)]

print(f"Genero completions per {len(test_samples)} prompt...\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    # Valuta
    task_type = sample["task_type"]
    difficulty = sample["difficulty"]
    reward = combined_reward(completion, task_type=task_type)

    print(f"--- [{task_type}/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

Genero completions per 4 prompt...

--- [python/hard] ---
Prompt:     Write a Python context manager class called `ResourcePool` that wraps a database transaction with au...
Completion: ```python
import psycopg2

class ResourcePool:
    def __init__(self, connection_string):
        self.connection_string = connection_string
        self.connection = None

    def __enter__(self):
  
Reward:     0.0

--- [json/hard] ---
Prompt:     Generate a deeply nested JSON object (at least 4 levels of nesting) representing a file system direc...
Completion: ```json
{
  "name": "Root",
  "id": 0,
  "children": [
    {
      "name": "Folder1",
      "id": 1,
      "children": [
        {
          "name": "FileA.txt",
          "id": 2,
          "children
Reward:     1.0

--- [json/medium] ---
Prompt:     Generate a JSON object representing a blog post with at least 8 fields, including one nested object ...
Completion: ```json
{
  "title": "Exploring the Tech Jungle: A Comprehensive Guide to Techno

## 6. Mini GRPO Training (pochi step di test)

Facciamo un training ridotto per verificare che la pipeline giri senza errori.

- **max_steps=20** — solo per test
- **num_generations=4** — minimo per avere varianza nelle reward
- **learning_rate=1e-6** — basso per evitare model collapse
- **beta=0.1** — KL penalty più alto per restare vicini al modello base
- **50 samples** — subset piccolo

In [27]:
from datasets import Dataset
from src.training.grpo_train import build_reward_fn

# Usa solo 50 samples per il test
MINI_N = 50
mini_ds = train_ds.select(range(min(MINI_N, len(train_ds))))  # type: ignore[arg-type]

# Formatta i prompt
formatted = []
for i in range(len(mini_ds)):
    s = mini_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Mini dataset: {len(prompt_dataset)} prompt pronti")

# Reward function
reward_fn = build_reward_fn(config["reward"])
print("Reward function creata ✓")

Mini dataset: 50 prompt pronti
Reward function creata ✓


In [ ]:
from trl.trainer.grpo_config import GRPOConfig

# Config leggera per test
mini_output_dir = str(ROOT / "experiments" / "checkpoints" / "grpo_test")
mini_log_dir = str(ROOT / "experiments" / "logs" / "grpo_test")
Path(mini_output_dir).mkdir(parents=True, exist_ok=True)
Path(mini_log_dir).mkdir(parents=True, exist_ok=True)

grpo_config = GRPOConfig(  # type: ignore[call-arg]
    output_dir=mini_output_dir,
    # Training ridotto
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    # GRPO
    num_generations=4,  # minimo per avere varianza con reward binarie
    max_completion_length=256,
    beta=0.1,  # KL penalty più alto per evitare divergenza
    temperature=0.7,
    # logging
    logging_dir=mini_log_dir,
    logging_steps=5,
    report_to="wandb"
)

print("GRPOConfig creata ✓")
print(f"  max_steps:          {grpo_config.max_steps}")
print(f"  num_generations:    {grpo_config.num_generations}")
print(f"  batch_size:         {grpo_config.per_device_train_batch_size}")
print(f"  grad_accum:         {grpo_config.gradient_accumulation_steps}")
print(f"  learning_rate:      {grpo_config.learning_rate}")
print(f"  beta (KL penalty):  {grpo_config.beta}")
print(f"  logging_steps:      {grpo_config.logging_steps}")
print(f"  report_to:          {grpo_config.report_to}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


GRPOConfig creata ✓
  max_steps:          20
  num_generations:    2
  batch_size:         1
  grad_accum:         2
  logging_steps:      5
  logging_dir:        /content/tris/experiments/logs/grpo_test
  report_to:          ['wandb']


In [29]:
from trl.trainer.grpo_trainer import GRPOTrainer

# Ricarica il modello fresco (il precedente potrebbe aver accumulato stato)
# Se hai poca VRAM, decommenta le 3 righe sotto per liberare memoria prima
# del model
torch.cuda.empty_cache()
model, tokenizer = load_model_and_tokenizer(config)

trainer = GRPOTrainer(  # type: ignore[arg-type]
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print("GRPOTrainer inizializzato ✓")
print("\nAvvio mini-training (20 step)...\n")

trainer.train()
print("\nMini-training completato ✓")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


GRPOTrainer inizializzato ✓

Avvio mini-training (20 step)...



wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cosmico445 (cosmico445-cosmic-inc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
5,0.000000
10,0.040965
15,0.000009
20,0.000004



Mini-training completato ✓


## 7. Verifica post-training

In [30]:
# Rigenera sugli stessi prompt di prima per vedere se qualcosa è cambiato
print("Completions dopo il mini-training:\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    task_type = sample["task_type"]
    difficulty = sample["difficulty"]
    reward = combined_reward(completion, task_type=task_type)

    print(f"--- [{task_type}/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Completions dopo il mini-training:

--- [python/hard] ---
Prompt:     Write a Python context manager class called `ResourcePool` that wraps a database transaction with au...
Completion: ```,1200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
Reward:     0.0

--- [json/hard] ---
Prompt:     Generate a deeply nested JSON object (at least 4 levels of nesting) representing a file system direc...
Completion: ```.1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
Reward:     0.0

--- [json/medium] ---
Prompt:     Generate a JSON object representing a blog post with at least 8 fields, including one nested object ...
Completion: ```.3000000000000000000000000000000000000000000000000000000000000000000000000000

In [31]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata ✓")

VRAM liberata ✓


## Riepilogo

Se tutte le celle sono andate a buon fine:

| Step | Status |
|------|--------|
| Imports & GPU | ✓ |
| Dataset generation | ✓ |
| Model loading (LoRA + 4bit) | ✓ |
| Reward functions | ✓ |
| Inference test | ✓ |
| GRPO mini-training | ✓ |
| Post-training check | ✓ |

**Prossimi passi:**
- Training completo: `uv run python -m src.training.grpo_train --config experiments/configs/grpo.yaml`
- Baseline eval: `uv run python -m src.evaluation.baseline_eval --config experiments/configs/baseline.yaml`
- SFT comparison: `uv run python -m src.training.sft_train --config experiments/configs/sft.yaml`